# Factor Model — Interview Project

**Goal:** build a factor analysis pipeline that decomposes portfolio returns into contributions from 8 macro risk factors, evaluates model performance, extracts statistical factors via PCA, and constructs a trend-following strategy.

The pipeline is evaluated on a held-out dataset with a different asset universe — all code is generic, no column names hard-coded.

| File | Content |
|---|---|
| `factors.csv` | Daily excess returns of 8 macro risk factors |
| `assets.csv` | Daily excess returns of 20 liquid futures |
| `portfolios.csv` | Daily excess returns of 4 multi-asset portfolios |


## 1. Loading


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from pathlib import Path

# ── Visual style ──────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.05)
plt.rcParams.update({
    "figure.dpi":        130,
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "axes.titlesize":    13,
    "axes.labelsize":    11,
})

ANNUALIZE = np.sqrt(252)   # daily-to-annual scaling factor

print("✓ Imports OK")

In [ ]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("data")

def read_returns(path):
    return pd.read_csv(path, index_col=0, parse_dates=True)

def load_dataset(data_path):
    data_path = Path(data_path)
    return {
        "factors":    read_returns(data_path / "factors.csv"),
        "assets":     read_returns(data_path / "assets.csv"),
        "portfolios": read_returns(data_path / "portfolios.csv"),
    }

def run_pipeline(data_path):
    dataset = load_dataset(data_path)
    factors = dataset["factors"]
    assets = dataset["assets"]
    portfolios = dataset["portfolios"]
    results = {"exposures": None, "performance": None, "pca": None, "trend": None}
    return results

dataset    = load_dataset(DATA_DIR)
factors    = dataset["factors"]
assets     = dataset["assets"]
portfolios = dataset["portfolios"]

print(f"Period: {factors.index.min().date()} to {factors.index.max().date()}")
print(f"Factors:    {factors.shape}")
print(f"Assets:     {assets.shape}")
print(f"Portfolios: {portfolios.shape}")


## 2. Exploration

This section covers data quality, distributional properties, correlation structure, regime dynamics, and stationarity. Every finding directly motivates a modelling choice in Section 3.


### 2.0 — Data Quality


In [ ]:
# ── Data Quality Check ────────────────────────────────────────────────────
print("=" * 60)
print("  DATA QUALITY CHECK")
print("=" * 60)

for name, df in [("Factors", factors), ("Assets", assets), ("Portfolios", portfolios)]:
    n_nan  = df.isna().sum().sum()
    n_dup  = df.index.duplicated().sum()
    n_miss = df.isna().any(axis=1).sum()
    pct    = 100 * n_nan / df.size
    print(f"\n── {name} {df.shape} ──")
    print(f"   Missing values  : {n_nan:>6}  ({pct:.3f}% of cells)")
    print(f"   Rows with NaN   : {n_miss:>6}")
    print(f"   Duplicate dates : {n_dup:>6}")
    print(f"   Date range      : {df.index.min().date()} → {df.index.max().date()}")

print("\n── NaN detail per asset ──")
nan_by_col = assets.isna().sum()
print(nan_by_col[nan_by_col > 0].to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Graphique 1 : nombre de NaN par asset ────────────────────────────────
nan_counts = assets.isna().sum().sort_values(ascending=True)
nan_counts = nan_counts[nan_counts > 0]

colors = ["#d73027" if v >= 10 else "#fc8d59" for v in nan_counts.values]
bars = axes[0].barh(nan_counts.index, nan_counts.values, color=colors, edgecolor="white")

for bar, val in zip(bars, nan_counts.values):
    axes[0].text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
                 str(val), va="center", fontsize=10, fontweight="bold")

axes[0].set_title("Missing values per asset", fontsize=13)
axes[0].set_xlabel("Number of missing values")
axes[0].axvline(x=5, color="gray", linestyle="--", alpha=0.5, label="threshold = 5")
axes[0].legend(fontsize=9)

# ── Panel 2: PB missing days by year ───────────────────────────────────────────────
pb_nan_dates = assets.index[assets["PB"].isna()]
year_counts  = pb_nan_dates.year.value_counts().sort_index()

axes[1].bar(year_counts.index, year_counts.values, color="#fc8d59", edgecolor="white", width=0.6)
for yr, cnt in year_counts.items():
    axes[1].text(yr, cnt + 0.1, str(cnt), ha="center", fontsize=10, fontweight="bold")

axes[1].set_title("PB — missing days per year", fontsize=13)
axes[1].set_xlabel("Year")
axes[1].set_ylabel("Missing days")
axes[1].set_xticks(year_counts.index)

plt.suptitle("Missing Value Analysis — Assets", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

`factors` and `portfolios` are complete. `assets` has 65 missing values: 15 contracts have exactly **1 missing day** each (isolated holidays), and `PB` (Pork Bellies) has **50 missing days** distributed consistently across all years — structural illiquidity of this contract.

**Decision:** impute with the cross-sectional median per day. Robust to concurrent large moves. With 0.118% of cells affected, the choice has no material impact on results.


In [ ]:
# ── Imputation : cross-sectional median per day ───────────────────────────
assets_clean = assets.apply(lambda row: row.fillna(row.median()), axis=1)

# Verification
print(f"NaN before : {assets.isna().sum().sum()}")
print(f"NaN after  : {assets_clean.isna().sum().sum()}")
print("✓ Imputation done")

### 2.1 — Descriptive Statistics


In [ ]:
# ── Annualised descriptive statistics – Factors ───────────────────────────
def descriptive_stats(df):
    mu    = df.mean()  * 252
    vol   = df.std()   * ANNUALIZE
    sr    = mu / vol
    skew  = df.skew()
    kurt  = df.kurt()
    
    stats = pd.DataFrame({
        "Ann. Return"  : mu.round(4),
        "Ann. Vol"     : vol.round(4),
        "Sharpe Ratio" : sr.round(3),
        "Skewness"     : skew.round(3),
        "Exc. Kurtosis": kurt.round(3),
    })
    return stats

stats_factors = descriptive_stats(factors)
print("=" * 65)
print("  DESCRIPTIVE STATISTICS — MACRO FACTORS (annualised)")
print("=" * 65)
print(stats_factors.to_string())

### 2.2 — Macro Factors: Risk / Return Profile


In [ ]:
# ── Risk / Return scatter — Macro Factors ────────────────────────────────
mu   = factors.mean()  * 252
vol  = factors.std()   * ANNUALIZE
sr   = mu / vol
kurt = factors.kurt()

# Manual offsets to avoid label overlap
label_offsets = {
    "equity":           (-60,   8),
    "interest_rate":    (  8, -14),
    "commodities":      (-90,   8),
    "credit":           (  8,   8),
    "emerging_market":  (  8, -14),
    "local_inflation":  (-80,   8),
    "foreign_currency": (  8,   8),
    "short_vol":        (  8,   8),
}

fig, ax = plt.subplots(figsize=(11, 7))

bubble_size = (kurt / kurt.max()) * 3000 + 200
scatter = ax.scatter(
    vol * 100, mu * 100,
    s=bubble_size, c=sr,
    cmap="RdYlGn", vmin=0, vmax=0.8,
    alpha=0.85, edgecolors="white", linewidths=1.5, zorder=3
)

for name in factors.columns:
    ox, oy = label_offsets[name]
    ax.annotate(
        name,
        xy=(vol[name] * 100, mu[name] * 100),
        xytext=(ox, oy), textcoords="offset points",
        fontsize=10, fontweight="bold",
        arrowprops=dict(arrowstyle="-", color="gray", lw=0.8)
            if abs(ox) > 10 or abs(oy) > 10 else None
    )

cbar = plt.colorbar(scatter, ax=ax, pad=0.02)
cbar.set_label("Sharpe Ratio", fontsize=10)

ax.axhline(0, color="black", linewidth=0.8, linestyle="--", alpha=0.4)
ax.axvline(vol.mean() * 100, color="gray", linewidth=0.8,
           linestyle="--", alpha=0.4, label="avg volatility")

for k_val, label in [(5, "kurtosis = 5"), (25, "kurtosis = 25"), (46, "kurtosis = 46")]:
    ax.scatter([], [], s=(k_val / kurt.max()) * 3000 + 200,
               color="gray", alpha=0.5, label=label)

ax.set_xlabel("Annualised Volatility (%)")
ax.set_ylabel("Annualised Return (%)")
ax.set_title("Macro Factors — Risk / Return Profile\n"
             "bubble size = excess kurtosis (fat tails)  |  color = Sharpe ratio",
             fontsize=12, fontweight="bold")
ax.legend(fontsize=9, loc="upper left")
ax.set_xlim(2, 16.5)
plt.tight_layout()
plt.show()

**Equity** is the best-compensated factor (Sharpe 0.69, ~10% p.a.). **Short_vol** looks attractive but the bubble size is a warning — kurtosis of 46 means selling volatility earns steadily until a crash wipes out years of gains overnight. **Credit** (Sharpe 0.63) is efficient at low vol. **Commodities** and **interest_rate** disappoint — a direct consequence of the commodity cycle and 2022 rate shock.

**Key insight:** Sharpe ratio alone is misleading for fat-tailed strategies. A model that only looks at means and variances will systematically underestimate tail risk.


### 2.3 — Cumulative Returns


In [ ]:
# ── Cumulative returns — Macro Factors ───────────────────────────────────
cumulative = (1 + factors).cumprod()

fig, ax = plt.subplots(figsize=(13, 6))

for col in cumulative.columns:
    ax.plot(cumulative.index, cumulative[col], linewidth=1.5, label=col)

# Highlight key market events
events = {
    "COVID\ncrash":   "2020-03-20",
    "Rate hike\ncycle": "2022-01-01",
}
for label, date in events.items():
    ax.axvline(pd.Timestamp(date), color="black",
               linewidth=0.8, linestyle="--", alpha=0.5)
    ax.text(pd.Timestamp(date), ax.get_ylim()[1] * 0.95,
            label, fontsize=8, ha="center",
            bbox=dict(boxstyle="round,pad=0.2", facecolor="white", alpha=0.7))

ax.axhline(1, color="black", linewidth=0.8, linestyle="-", alpha=0.3)
ax.set_title("Macro Factors — Cumulative Returns (2015–2025)",
             fontsize=13, fontweight="bold")
ax.set_ylabel("Growth of $1")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.legend(loc="upper left", fontsize=9, ncol=2)
plt.tight_layout()
plt.show()

In [ ]:
# ── Portfolios — Performance Overview ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# ── Graphique 1 : Cumulative returns ─────────────────────────────────────
colors     = ["#2ecc71", "#e74c3c", "#3498db", "#f39c12"]
cumulative = (1 + portfolios).cumprod()

for col, color in zip(portfolios.columns, colors):
    axes[0].plot(cumulative.index, cumulative[col],
                 linewidth=1.8, color=color,
                 label=col.replace("portfolio_", ""))

events = {"COVID\ncrash": "2020-03-20", "Rate hike\ncycle": "2022-01-01"}
for label, date in events.items():
    axes[0].axvline(pd.Timestamp(date), color="black",
                    linewidth=0.8, linestyle="--", alpha=0.5)
    axes[0].text(pd.Timestamp(date), cumulative.max().max() * 0.97,
                 label, fontsize=8, ha="center",
                 bbox=dict(boxstyle="round,pad=0.2", facecolor="white", alpha=0.7))

axes[0].axhline(1, color="black", linewidth=0.6, alpha=0.2)
axes[0].set_title("Cumulative Returns", fontsize=12, fontweight="bold")
axes[0].set_ylabel("Growth of $1")
axes[0].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
axes[0].legend(fontsize=9)

# ── Graphique 2 : Return / Vol / Sharpe ──────────────────────────────────
port_labels = [c.replace("portfolio_", "") for c in portfolios.columns]
mu  = portfolios.mean() * 252
vol = portfolios.std()  * ANNUALIZE
sr  = mu / vol

x     = np.arange(len(port_labels))
width = 0.25

for bars, vals, label, color in [
    (x - width, mu  * 100, "Ann. Return (%)",  "#3498db"),
    (x,         vol * 100, "Ann. Vol (%)",      "#e67e22"),
    (x + width, sr,        "Sharpe Ratio",      "#2ecc71"),
]:
    b = axes[1].bar(bars, vals, width, label=label,
                    color=color, alpha=0.85, edgecolor="white")
    for bar in b:
        h = bar.get_height()
        axes[1].text(bar.get_x() + bar.get_width() / 2,
                     h + 0.1 if h >= 0 else h - 0.4,
                     f"{h:.2f}", ha="center", fontsize=7.5, fontweight="bold")

axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_xticks(x)
axes[1].set_xticklabels(port_labels, rotation=15, ha="right", fontsize=9)
axes[1].set_title("Return / Vol / Sharpe Comparison", fontsize=12, fontweight="bold")
axes[1].legend(fontsize=9)

plt.suptitle("Portfolios — Performance Overview (2015–2025)",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

The sample covers three distinct regimes: a low-vol bull market (2015–2019), the COVID crash and recovery (2020–2021), and the rate hike shock (2022). Factor behaviour changes materially across them — a static model averages them out. `trend` is flat until 2022 then surges; `inverse_vol` bleeds as its bond-heavy positioning gets crushed.


### 2.4 — Factor Correlation Structure


In [ ]:
# ── Correlation matrix — Macro Factors ───────────────────────────────────
corr = factors.corr()

fig, ax = plt.subplots(figsize=(9, 7))

# Lower triangle only — upper is symmetric
mask_upper = np.triu(np.ones_like(corr, dtype=bool), k=1)

sns.heatmap(
    corr,
    mask=mask_upper,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    center=0,
    vmin=-1, vmax=1,
    linewidths=0.5,
    ax=ax,
    annot_kws={"size": 10}
)

ax.set_title("Macro Factors — Pairwise Correlation Matrix",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

`equity`, `short_vol`, `credit`, and `emerging_market` form a correlated cluster (0.37–0.85). They all measure global risk appetite. `foreign_currency` is nearly orthogonal — a genuine diversifier. `interest_rate` has a mild negative correlation with equity (-0.05), though this breaks down in 2022.

**For regression:** this collinearity will inflate OLS standard errors. Ridge regularisation is the remedy.


### 2.5 — Regime-Conditional Correlations


In [ ]:
# ── Regime analysis — rolling correlation ─────────────────────────────────
REGIMES = {
    "Pre-COVID\n(2015-2019)":   ("2015-01-01", "2019-12-31"),
    "COVID crash\n(2020 Q1)":   ("2020-01-01", "2020-03-31"),
    "Recovery\n(2020 Q2-2021)": ("2020-04-01", "2021-12-31"),
    "Rate hike\n(2022)":        ("2022-01-01", "2022-12-31"),
    "Post-hike\n(2023-2025)":   ("2023-01-01", "2025-12-31"),
}

# Three pairs with most interesting regime dynamics
pairs = [
    ("equity",    "short_vol",     "equity / short_vol"),
    ("equity",    "interest_rate", "equity / interest_rate"),
    ("equity",    "credit",        "equity / credit"),
]

# Compute pairwise correlation within each regime
regime_results = {label: {} for _, _, label in pairs}

for regime_name, (start, end) in REGIMES.items():
    sub = factors.loc[start:end]
    for f1, f2, label in pairs:
        regime_results[label][regime_name] = sub[f1].corr(sub[f2])

# ── Plot ──────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))

x          = np.arange(len(REGIMES))
width      = 0.25
colors     = ["#4C72B0", "#DD8452", "#55A868"]
regime_labels = list(REGIMES.keys())

for i, (_, _, label) in enumerate(pairs):
    values = [regime_results[label][r] for r in regime_labels]
    bars   = ax.bar(x + i * width, values, width,
                    label=label, color=colors[i],
                    alpha=0.85, edgecolor="white")
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.01 if val >= 0 else bar.get_height() - 0.06,
                f"{val:.2f}", ha="center", fontsize=8, fontweight="bold")

ax.axhline(0, color="black", linewidth=0.8)
ax.set_xticks(x + width)
ax.set_xticklabels(regime_labels, fontsize=9)
ax.set_ylabel("Pearson Correlation")
ax.set_title("Factor Correlations Across Market Regimes",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=9)
ax.set_ylim(-0.6, 1.1)
plt.tight_layout()
plt.show()

The `equity / short_vol` correlation is stable and high (0.80–0.95) in all regimes — effectively the same risk factor.

More important: `equity / interest_rate` is **negative** in normal times (flight-to-quality) but turns **positive** in 2022 (rate hike crushed both equities and bonds). A static model misses both regimes. This is the strongest argument for rolling estimation.


### 2.6 — Rolling Volatility


In [ ]:
# ── Rolling EWMA volatility — Macro Factors ───────────────────────────────
ewma_vol = factors.ewm(span=60, min_periods=20).std() * ANNUALIZE

fig, axes = plt.subplots(4, 2, figsize=(14, 12), sharex=True)
axes = axes.flatten()

# Key events to highlight
events = {
    "COVID": "2020-03-20",
    "Rate hike": "2022-01-01",
}

for i, col in enumerate(factors.columns):
    ax = axes[i]
    ax.plot(ewma_vol.index, ewma_vol[col] * 100,
            linewidth=1.5, color="#4C72B0")
    ax.fill_between(ewma_vol.index, ewma_vol[col] * 100,
                    alpha=0.15, color="#4C72B0")

    for label, date in events.items():
        ax.axvline(pd.Timestamp(date), color="red",
                   linewidth=0.8, linestyle="--", alpha=0.6)

    ax.set_title(col, fontsize=11, fontweight="bold")
    ax.set_ylabel("Vol (%)", fontsize=9)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

fig.suptitle("Macro Factors — Rolling EWMA Volatility (span=60d)",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

All factors show pronounced volatility clustering. `short_vol` peaks at ~60% annualised in March 2020 — roughly 6× its normal level.

Two consequences: (1) OLS residuals will be heteroskedastic; (2) crisis observations carry more noise than signal. Both are addressed by **Weighted Least Squares** in Section 3.


### 2.7 — Stationarity (ADF + KPSS)


In [ ]:
# ── Stationarity test — ADF + KPSS ───────────────────────────────────────
from statsmodels.tsa.stattools import adfuller, kpss
import warnings

def stationarity_tests(df, name):
    print(f"\n{'=' * 70}")
    print(f"  STATIONARITY TESTS — {name}")
    print(f"{'=' * 70}")
    print(f"{'Series':<22} {'ADF p-val':>10} {'KPSS p-val':>11} {'Stationary?':>12}")
    print("-" * 58)
    for col in df.columns:
        # ADF: H0 = non-stationary
        adf_res  = adfuller(df[col].dropna(), autolag="AIC")
        adf_p    = adf_res[1]
        # KPSS: H0 = stationary
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            kpss_res = kpss(df[col].dropna(), regression='c', nlags='auto')
        kpss_p = kpss_res[1]
        # Stationary if both hold
        flag = "✓ Yes" if (adf_p < 0.05 and kpss_p > 0.05) else "✗ No"
        print(f"{col:<22} {adf_p:>10.2e} {kpss_p:>11.4f} {flag:>12}")

stationarity_tests(factors,    "MACRO FACTORS")
stationarity_tests(portfolios, "PORTFOLIOS")

All return series are stationary — daily returns oscillate around zero with no persistent drift. Regression in levels is valid, no differencing needed. `portfolio_trend` has a KPSS p-value of 0.095 (marginally above threshold), consistent with the mild persistence of a trend-following strategy.


## 3. Factor Model

$$r_t = f_\\theta(\\mathbf{x}_t) + \\varepsilon_t$$

Five models of increasing sophistication, each addressing a specific problem identified by the previous one.

| Model | Problem addressed |
|---|---|
| OLS | Baseline — identifies issues |
| Ridge-WLS | Multicollinearity + heteroskedasticity |
| LASSO + features | Non-linear regime effects |
| Rolling OLS | Time-varying exposures |
| Huber | Observation-level outliers (negative result) |

All models are generic: no asset or portfolio names hard-coded.


### 3.1 — Baseline OLS


In [ ]:
# ── Baseline OLS — one model per portfolio ────────────────────────────────
import statsmodels.api as sm

# Align factors and portfolios on common dates
common_idx = factors.index.intersection(portfolios.index)
F = factors.loc[common_idx]
P = portfolios.loc[common_idx]

# Add constant for intercept
X_ols = sm.add_constant(F)

ols_results = {}
for port in P.columns:
    model  = sm.OLS(P[port], X_ols).fit()
    ols_results[port] = model

# ── Summary table ─────────────────────────────────────────────────────────
print("=" * 65)
print("  BASELINE OLS — R² per portfolio")
print("=" * 65)
for port, res in ols_results.items():
    name = port.replace("portfolio_", "")
    print(f"  {name:<25}  R² = {res.rsquared:.3f}   "
          f"Adj R² = {res.rsquared_adj:.3f}")

print("\n── Coefficients (equal_weight) ──")
print(ols_results["portfolio_equal_weight"].summary2().tables[1].round(3))

In [ ]:
# ── VIF — Variance Inflation Factor ──────────────────────────────────────
from statsmodels.stats.outliers_influence import variance_inflation_factor

vif_data = pd.DataFrame()
vif_data["Factor"] = F.columns
vif_data["VIF"]    = [variance_inflation_factor(F.values, i)
                      for i in range(F.shape[1])]
vif_data = vif_data.sort_values("VIF", ascending=False)

# VIF > 5: moderate | VIF > 10: severe
vif_data["Problem?"] = vif_data["VIF"].apply(
    lambda x: "🔴 Severe"   if x > 10
         else "🟡 Moderate" if x > 5
         else "🟢 OK"
)

print("=" * 50)
print("  VARIANCE INFLATION FACTOR (VIF)")
print("=" * 50)
print(vif_data.to_string(index=False))
print("\nRule of thumb: VIF > 5 = moderate, VIF > 10 = severe")

R² is high for passive portfolios (0.835, 0.623) and low for trend strategies (0.160, 0.133). The gap reflects fundamentally different portfolio construction — not a modelling failure.

VIF confirms moderate multicollinearity on `equity` (6.2), driven by its correlation with `short_vol` and `emerging_market`. Ridge regularisation is the appropriate remedy.


### 3.2 — WLS Weighting Scheme


In [ ]:
# ── WLS weights — inverse EWMA variance ──────────────────────────────────

# Composite factor vol: cross-sectional std, smoothed with EWMA
factor_composite_vol = F.std(axis=1).ewm(span=60, min_periods=20).mean()

# Backfill warm-up NaNs
factor_composite_vol = factor_composite_vol.bfill()

# Weights = inverse vol: calm days up-weighted, turbulent days down-weighted
wls_weights = 1.0 / factor_composite_vol.clip(
    lower=factor_composite_vol.quantile(0.05)  # clip to avoid division by near-zero
)
wls_weights = wls_weights / wls_weights.mean()  # normalise to mean = 1

# ── Visualisation des poids ───────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True)

axes[0].plot(factor_composite_vol.index,
             factor_composite_vol * 100 * np.sqrt(252),
             linewidth=1.2, color="#3498db")
axes[0].fill_between(factor_composite_vol.index,
                     factor_composite_vol * 100 * np.sqrt(252),
                     alpha=0.15, color="#3498db")
axes[0].set_title("Composite Factor Volatility (EWMA span=60d)",
                  fontsize=11, fontweight="bold")
axes[0].set_ylabel("Ann. Vol (%)")

axes[1].plot(wls_weights.index, wls_weights,
             linewidth=1.2, color="#e67e22")
axes[1].fill_between(wls_weights.index, wls_weights,
                     alpha=0.15, color="#e67e22")
axes[1].axhline(1, color="black", linewidth=0.8,
                linestyle="--", alpha=0.5, label="average weight = 1")
axes[1].set_title("WLS Weights (inverse volatility)",
                  fontsize=11, fontweight="bold")
axes[1].set_ylabel("Weight")
axes[1].legend(fontsize=9)
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

plt.suptitle("WLS Weighting Scheme", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

print(f"\nWeight stats:")
print(f"  Mean   : {wls_weights.mean():.3f}")
print(f"  COVID  : {wls_weights.loc['2020-03-20']:.3f}")
print(f"  Normal : {wls_weights.loc['2018-01-02']:.3f}")

During the COVID crash the WLS weight drops to ~0.3 — that quarter contributes one-third as much as a calm day to the regression. This prevents a single extreme event from dominating a decade of coefficient estimates.


### 3.3 — Ridge-WLS


In [ ]:
# ── Ridge-WLS — Final Model ───────────────────────────────────────────────
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
import warnings
warnings.filterwarnings("ignore")

# TimeSeriesSplit respects temporal ordering — no look-ahead bias
tscv        = TimeSeriesSplit(n_splits=5)
alphas_grid = np.logspace(-4, 5, 200)

# Alpha floor: enforce minimum regularisation for OOS stability
ALPHA_FLOOR = 1.0

ridge_results = {}

for port in P.columns:
    y = P[port].values
    X = F.values
    w = wls_weights.values

    scaler   = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Manual CV with TimeSeriesSplit
    cv_scores = []
    for a in alphas_grid:
        ridge = Ridge(alpha=a, fit_intercept=True)
        fold_scores = []
        for train_idx, val_idx in tscv.split(X_scaled):
            ridge.fit(X_scaled[train_idx], y[train_idx],
                      sample_weight=w[train_idx])
            y_pred = ridge.predict(X_scaled[val_idx])
            fold_scores.append(np.mean((y[val_idx] - y_pred) ** 2))
        cv_scores.append(np.mean(fold_scores))

    # Apply alpha floor
    best_alpha = max(alphas_grid[np.argmin(cv_scores)], ALPHA_FLOOR)

    ridge = Ridge(alpha=best_alpha, fit_intercept=True)
    ridge.fit(X_scaled, y, sample_weight=w)

    beta      = ridge.coef_ / scaler.scale_
    alpha_hat = ridge.intercept_ - (
        scaler.mean_ / scaler.scale_ * ridge.coef_
    ).sum()
    y_hat     = X @ beta + alpha_hat
    resid     = y - y_hat
    r2        = 1 - np.sum(resid**2) / np.sum((y - y.mean())**2)

    ridge_results[port] = {
        "alpha":       alpha_hat,
        "betas":       dict(zip(F.columns, beta)),
        "r2":          r2,
        "residuals":   resid,
        "y_hat":       y_hat,
        "ridge_alpha": best_alpha,
    }

print("=" * 65)
print("  RIDGE-WLS — Final Results")
print("=" * 65)
print(f"\n{'Portfolio':<28} {'R²':>6}  {'Ridge α':>10}")
print("-" * 48)
for port, res in ridge_results.items():
    name = port.replace("portfolio_", "")
    print(f"  {name:<26} {res['r2']:>6.3f}  {res['ridge_alpha']:>10.2f}")

### 3.4 — Factor Exposures


In [ ]:
# ── Factor exposures heatmap ──────────────────────────────────────────────
betas_df = pd.DataFrame(
    {port: res["betas"] for port, res in ridge_results.items()}
).T
betas_df.index = [i.replace("portfolio_", "") for i in betas_df.index]

fig, ax = plt.subplots(figsize=(12, 4))
sns.heatmap(
    betas_df,
    annot=True,
    fmt=".3f",
    cmap="RdBu_r",
    center=0,
    linewidths=0.5,
    ax=ax,
    annot_kws={"size": 10}
)
ax.set_title("Ridge-WLS Factor Exposures (β) per Portfolio",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Macro Factor")
ax.set_ylabel("Portfolio")
plt.tight_layout()
plt.show()

**equal_weight** is dominated by `commodities` (β=0.46) — 13 of 20 futures are commodity contracts. **inverse_vol** loads on `interest_rate` (β=0.31) — inverse-vol weighting overweights low-vol assets, which here means fixed income. **trend** and **trend_neutral** have near-zero betas everywhere — these strategies generate alpha through dynamic positioning, not static factor loading.


### 3.5 — LASSO with Advanced Features

Three economically motivated, **fully generic** transformations:

1. **Rolling Z-score (63d):** normalises by recent context. A +2% equity return in calm 2017 differs from +2% in March 2020.
2. **Squared returns of top-2 kurtosis factors:** non-linear tail exposure. Identified automatically.
3. **Interaction of most correlated pair:** stress regime capture. Identified automatically.

With 19 features, LASSO’s automatic feature selection zeroes out irrelevant inputs and keeps the model interpretable.


In [ ]:
# ── Advanced feature engineering — fully generic ─────────────────────────

# Z-score (63d): normalises by recent context, fully generic
ZSCORE_WIN    = 63
roll_mean     = F.rolling(ZSCORE_WIN, min_periods=20).mean()
roll_std      = F.rolling(ZSCORE_WIN, min_periods=20).std()
factor_zscore = ((F - roll_mean) / roll_std)
factor_zscore.columns = [c + "_zscore" for c in F.columns]

# Squared returns of top-2 kurtosis factors: non-linear tail exposure, generic
top2_kurt = F.kurt().nlargest(2).index.tolist()
fat_tail_sq = F[top2_kurt] ** 2
fat_tail_sq.columns = [c + "_sq" for c in top2_kurt]

print(f"Top-2 kurtosis factors identified : {top2_kurt}")

# Interaction of most correlated pair: stress regime capture, generic
corr_abs = F.corr().abs()
np.fill_diagonal(corr_abs.values, 0)   # ignore self-correlation
idx = np.unravel_index(corr_abs.values.argmax(), corr_abs.shape)
col_a, col_b = F.columns[idx[0]], F.columns[idx[1]]
interaction  = pd.DataFrame(
    {f"{col_a}_x_{col_b}": F[col_a] * F[col_b]},
    index=F.index
)

print(f"Most correlated pair identified   : {col_a} × {col_b} "
      f"(corr={corr_abs.loc[col_a, col_b]:.3f})")

# Combine
F_advanced = pd.concat([
    F,
    factor_zscore,
    fat_tail_sq,
    interaction
], axis=1)

# Drop warm-up NaNs
valid_idx = F_advanced.dropna().index
F_adv     = F_advanced.loc[valid_idx]
P_adv     = P.loc[valid_idx]
wls_adv   = wls_weights.loc[valid_idx]

print(f"\nFeature breakdown:")
print(f"  Raw factors        :  {F.shape[1]}")
print(f"  Z-scores           :  {F.shape[1]}")
print(f"  Fat-tail squared   :  2")
print(f"  Stress interaction :  1")
print(f"  Total              :  {F_adv.shape[1]}")
print(f"\nRows after dropna  : {len(F_adv)} "
      f"(lost {len(F) - len(F_adv)} warm-up rows)")

In [ ]:
# ── LassoCV on advanced features ─────────────────────────────────────────
# LASSO: automatic feature selection (zeroes irrelevant betas)
# Preferred over Ridge for sparse high-dimensional inputs
from sklearn.linear_model import LassoCV

tscv        = TimeSeriesSplit(n_splits=5)
alphas_lasso = np.logspace(-6, 1, 100)

lasso_advanced = {}

for port in P_adv.columns:
    y = P_adv[port].values
    X = F_adv.values
    w = wls_adv.values

    scaler   = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    lasso = LassoCV(alphas=alphas_lasso, cv=tscv,
                    fit_intercept=True, max_iter=10000)
    lasso.fit(X_scaled, y, sample_weight=w)

    beta      = lasso.coef_ / scaler.scale_
    alpha_hat = lasso.intercept_ - (
        scaler.mean_ / scaler.scale_ * lasso.coef_
    ).sum()
    y_hat = X @ beta + alpha_hat
    resid = y - y_hat
    r2    = 1 - np.sum(resid**2) / np.sum((y - y.mean())**2)

    # Count non-zero features selected by LASSO
    n_selected = np.sum(lasso.coef_ != 0)

    lasso_advanced[port] = {
        "alpha":       alpha_hat,
        "betas":       dict(zip(F_adv.columns, beta)),
        "r2":          r2,
        "residuals":   resid,
        "y_hat":       y_hat,
        "lasso_alpha": lasso.alpha_,
        "n_selected":  n_selected,
    }

# ── Comparison table ──────────────────────────────────────────────────────
print("=" * 72)
print("  MODEL COMPARISON : OLS vs Ridge-WLS vs LASSO Advanced")
print("=" * 72)
print(f"\n{'Portfolio':<22} {'OLS':>6} {'Ridge':>8} "
      f"{'LASSO+':>8} {'Gain':>7} {'Features used':>15}")
print("-" * 70)

import statsmodels.api as sm
for port in P_adv.columns:
    name = port.replace("portfolio_", "")

    X_ols  = sm.add_constant(F.values)
    r2_ols = sm.OLS(P[port].values, X_ols).fit().rsquared

    r2_ridge = ridge_results[port]["r2"]
    r2_lasso = lasso_advanced[port]["r2"]
    gain     = r2_lasso - r2_ridge
    n_sel    = lasso_advanced[port]["n_selected"]

    print(f"  {name:<20} {r2_ols:>6.3f} {r2_ridge:>8.3f} "
          f"{r2_lasso:>8.3f} {gain:>+7.3f} {n_sel:>8}/{F_adv.shape[1]}")

In [ ]:
# ── LASSO feature importance — which features were selected ───────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

for i, port in enumerate(P.columns):
    name   = port.replace("portfolio_", "")
    betas  = lasso_advanced[port]["betas"]

    # Keep only non-zero betas
    betas_nonzero = {k: v for k, v in betas.items() if abs(v) > 1e-8}
    betas_zero    = {k: v for k, v in betas.items() if abs(v) <= 1e-8}

    # Sort by absolute value
    sorted_betas = dict(sorted(betas_nonzero.items(),
                               key=lambda x: abs(x[1]),
                               reverse=True))

    labels = list(sorted_betas.keys())
    values = list(sorted_betas.values())
    colors = ["#e74c3c" if v < 0 else "#2ecc71" for v in values]

    axes[i].barh(labels, values, color=colors,
                 edgecolor="white", alpha=0.85)
    axes[i].axvline(0, color="black", linewidth=0.8)
    axes[i].set_title(
        f"{name}\n"
        f"R²={lasso_advanced[port]['r2']:.3f}  "
        f"|  {len(betas_nonzero)}/{len(betas)} features selected  "
        f"|  {len(betas_zero)} set to zero",
        fontsize=10, fontweight="bold"
    )
    axes[i].set_xlabel("Beta")

    # Add value labels
    for bar, val in zip(axes[i].patches, values):
        axes[i].text(
            val + 0.0001 if val >= 0 else val - 0.0001,
            bar.get_y() + bar.get_height() / 2,
            f"{val:.4f}", va="center",
            ha="left" if val >= 0 else "right",
            fontsize=7.5
        )

plt.suptitle("LASSO+ — Selected Features per Portfolio\n"
             "green = positive exposure  |  red = negative exposure  "
             "|  absent = set to zero by LASSO",
             fontsize=12, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

# Summary table
print("\n=== Features eliminated by LASSO (set to zero) ===\n")
for port in P.columns:
    name  = port.replace("portfolio_", "")
    zeros = [k for k, v in lasso_advanced[port]["betas"].items()
             if abs(v) <= 1e-8]
    print(f"  {name:<22} eliminated: {', '.join(zeros)}")

Gains concentrate where expected: passive portfolios see minimal improvement (raw factors already sufficient), while trend portfolios improve substantially (`trend` +9.4%, `trend_neutral` +4.9%). Z-scores better capture how trend strategies respond to factor moves relative to recent history.

For `trend_neutral`, LASSO retains only 3 of 19 features — all Z-scores. The portfolio responds to whether rates/credit are high *relative to recent history*, not to their absolute level. For `trend`, every selected beta is negative: the strategy systematically short-sells credit and vol risk.


### 3.6 — Rolling Window OLS

A static model assumes exposures are constant. Section 2.5 showed they are not. Rolling OLS re-estimates betas daily using only the most recent 252 days. Crucially, this is used here as a *predictive model* (betas from t-252 to t-1 predict day t), not just a diagnostic tool.


In [ ]:
# ── Rolling Window OLS — time-varying factor exposures ───────────────────
ROLL_WIN = 252

rolling_ols = {}

for port in P.columns:
    y      = P[port].values
    X      = F.values
    n      = len(y)
    y_hat  = np.full(n, np.nan)
    betas  = np.full((n, F.shape[1]), np.nan)

    for t in range(ROLL_WIN, n):
        # Estimate on past ROLL_WIN days only — no look-ahead
        X_win  = np.column_stack([np.ones(ROLL_WIN),
                                   X[t - ROLL_WIN:t]])
        y_win  = y[t - ROLL_WIN:t]
        coef, *_ = np.linalg.lstsq(X_win, y_win, rcond=None)

        # Predict day t using yesterday's betas
        y_hat[t]    = np.dot(np.array([1] + list(X[t])), coef)
        betas[t, :] = coef[1:]

    # R² on valid predictions only (after warm-up)
    valid   = ~np.isnan(y_hat)
    r2_roll = (1 - np.sum((y[valid] - y_hat[valid])**2) /
                   np.sum((y[valid] - y[valid].mean())**2))

    rolling_ols[port] = {
        "y_hat":  y_hat,
        "betas":  pd.DataFrame(betas, index=F.index,
                               columns=F.columns),
        "r2":     r2_roll,
    }

# ── Results ───────────────────────────────────────────────────────────────
print("=" * 60)
print("  ROLLING OLS — R² (out-of-sample, 252-day window)")
print("=" * 60)
print(f"\n{'Portfolio':<26} {'R² static OLS':>14} {'R² rolling':>11}")
print("-" * 53)
for port in P.columns:
    name     = port.replace("portfolio_", "")
    r2_stat  = ols_results[port].rsquared
    r2_roll  = rolling_ols[port]["r2"]
    better   = "✓ better" if r2_roll > r2_stat else "✗ worse"
    print(f"  {name:<24} {r2_stat:>14.3f} {r2_roll:>11.3f}  {better}")

Rolling OLS outperforms static OLS on 3 of 4 portfolios. The largest gain is on `trend_neutral` (+0.114) — its factor exposures shift dramatically between regimes and a 252-day window adapts within a trading year. `equal_weight` is unchanged: structurally stable exposures make full-sample estimation preferable.


### 3.7 — Huber Robust Regression

WLS down-weights volatile *periods*; Huber down-weights extreme *individual observations*. Given fat-tailed residuals, the motivation is clear. The null hypothesis is that extreme residuals are noise.


In [ ]:
# ── Huber Robust Regression ───────────────────────────────────────────────
from sklearn.linear_model import HuberRegressor
from sklearn.preprocessing import StandardScaler

huber_results = {}

for port in P.columns:
    y = P[port].values
    X = F.values

    scaler   = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Epsilon controls the threshold between quadratic and linear loss
    # We try a range and pick the best via TimeSeriesSplit CV
    epsilons  = [1.1, 1.35, 1.5, 1.75, 2.0, 2.5, 3.0]
    tscv      = TimeSeriesSplit(n_splits=5)
    best_eps  = 1.35
    best_mse  = np.inf

    for eps in epsilons:
        fold_scores = []
        for train_idx, val_idx in tscv.split(X_scaled):
            huber = HuberRegressor(epsilon=eps, max_iter=500)
            huber.fit(X_scaled[train_idx], y[train_idx])
            y_pred = huber.predict(X_scaled[val_idx])
            fold_scores.append(np.mean((y[val_idx] - y_pred)**2))
        mse = np.mean(fold_scores)
        if mse < best_mse:
            best_mse = mse
            best_eps = eps

    # Fit final model with best epsilon
    huber_final = HuberRegressor(epsilon=best_eps, max_iter=500)
    huber_final.fit(X_scaled, y)

    # Unscale coefficients
    beta      = huber_final.coef_ / scaler.scale_
    alpha_hat = huber_final.intercept_ - (
        scaler.mean_ / scaler.scale_ * huber_final.coef_
    ).sum()

    y_hat = X @ beta + alpha_hat
    resid = y - y_hat
    r2    = 1 - np.sum(resid**2) / np.sum((y - y.mean())**2)

    huber_results[port] = {
        "alpha":       alpha_hat,
        "betas":       dict(zip(F.columns, beta)),
        "r2":          r2,
        "residuals":   resid,
        "y_hat":       y_hat,
        "epsilon":     best_eps,
    }

print("=" * 65)
print("  HUBER ROBUST REGRESSION — Results")
print("=" * 65)
print(f"\n{'Portfolio':<26} {'OLS R²':>8} {'Huber R²':>10} "
      f"{'Gain':>8} {'Best ε':>8}")
print("-" * 62)
for port in P.columns:
    name    = port.replace("portfolio_", "")
    r2_ols  = ols_results[port].rsquared
    r2_hub  = huber_results[port]["r2"]
    gain    = r2_hub - r2_ols
    eps     = huber_results[port]["epsilon"]
    print(f"  {name:<24} {r2_ols:>8.3f} {r2_hub:>10.3f} "
          f"{gain:>+8.3f} {eps:>8.2f}")

No improvement — and `trend_neutral` degrades slightly (-0.011). The largest residuals cluster in March 2020: COVID crash days where the strategy moved dramatically. These are not noise; they are the most informative observations about how the strategy responds to extreme stress. Downweighting them removes signal, not noise.

**Negative result:** robust statistics are appropriate when outliers are measurement errors, not genuine market events. Huber is not used in the final pipeline.


### 3.8 — Out-of-Sample Validation

All in-sample results are optimistic by construction. We fit each model on 2015–2022 and evaluate on 2023–2025 — replicating what the evaluators will do on the held-out dataset.


In [ ]:
# ── Out-of-sample validation — train 2015-2022, test 2023-2025 ────────────
TRAIN_END = "2022-12-31"
TEST_START = "2023-01-01"

# Split
train_idx = F.index[F.index <= TRAIN_END]
test_idx  = F.index[F.index >= TEST_START]

F_train = F.loc[train_idx];  F_test = F.loc[test_idx]
P_train = P.loc[train_idx];  P_test = P.loc[test_idx]
w_train = wls_weights.loc[train_idx]

print(f"Training : {train_idx.min().date()} → {train_idx.max().date()} "
      f"({len(train_idx)} days)")
print(f"Test     : {test_idx.min().date()}  → {test_idx.max().date()} "
      f"({len(test_idx)} days)\n")

def oos_r2(y_true, y_pred):
    ss_r = np.sum((y_true - y_pred)**2)
    ss_t = np.sum((y_true - y_true.mean())**2)
    return 1 - ss_r / ss_t

oos_results = {port: {} for port in P.columns}

for port in P.columns:
    y_train = P_train[port].values
    y_test  = P_test[port].values
    w_tr    = w_train.values

    # ── OLS ───────────────────────────────────────────────────────────
    X_tr = sm.add_constant(F_train.values)
    X_te = sm.add_constant(F_test.values)
    ols  = sm.OLS(y_train, X_tr).fit()
    oos_results[port]["OLS"] = oos_r2(y_test, ols.predict(X_te))

    # ── Ridge-WLS ─────────────────────────────────────────────────────
    scaler_r  = StandardScaler()
    Xtr_r     = scaler_r.fit_transform(F_train.values)
    Xte_r     = scaler_r.transform(F_test.values)
    tscv      = TimeSeriesSplit(n_splits=5)
    cv_scores = []
    for a in np.logspace(-4, 5, 100):
        ridge = Ridge(alpha=a, fit_intercept=True)
        folds = []
        for tri, vli in tscv.split(Xtr_r):
            ridge.fit(Xtr_r[tri], y_train[tri], sample_weight=w_tr[tri])
            folds.append(np.mean((y_train[vli] - ridge.predict(Xtr_r[vli]))**2))
        cv_scores.append(np.mean(folds))
    best_a = max(np.logspace(-4, 5, 100)[np.argmin(cv_scores)], 1.0)
    ridge  = Ridge(alpha=best_a, fit_intercept=True)
    ridge.fit(Xtr_r, y_train, sample_weight=w_tr)
    beta      = ridge.coef_ / scaler_r.scale_
    alpha_hat = ridge.intercept_ - (scaler_r.mean_ / scaler_r.scale_ * ridge.coef_).sum()
    oos_results[port]["Ridge-WLS"] = oos_r2(y_test, F_test.values @ beta + alpha_hat)

    # ── LASSO+ ────────────────────────────────────────────────────────
    # Build advanced features on train, transform test
    def build_features(F_in, F_ref=None):
        ref   = F_ref if F_ref is not None else F_in
        zwin  = 63
        rmean = ref.rolling(zwin, min_periods=20).mean().iloc[-1]
        rstd  = ref.rolling(zwin, min_periods=20).std().iloc[-1]
        zscore = (F_in - rmean) / rstd
        zscore.columns = [c + "_zscore" for c in F_in.columns]
        top2k  = ref.kurt().nlargest(2).index
        sq     = F_in[top2k] ** 2
        sq.columns = [c + "_sq" for c in top2k]
        corr_abs = ref.corr().abs()
        np.fill_diagonal(corr_abs.values, 0)
        idx    = np.unravel_index(corr_abs.values.argmax(), corr_abs.shape)
        ca, cb = F_in.columns[idx[0]], F_in.columns[idx[1]]
        inter  = pd.DataFrame({f"{ca}_x_{cb}": F_in[ca] * F_in[cb]}, index=F_in.index)
        return pd.concat([F_in, zscore, sq, inter], axis=1).dropna()

    F_adv_tr = build_features(F_train)
    # Align test features using training stats
    F_adv_te = build_features(F_test, F_ref=F_train)

    y_tr_adv  = P_train.loc[F_adv_tr.index, port].values
    scaler_l  = StandardScaler()
    Xtr_l     = scaler_l.fit_transform(F_adv_tr.values)
    Xte_l     = scaler_l.transform(F_adv_te.values)
    lasso     = LassoCV(alphas=np.logspace(-6, 1, 50),
                        cv=tscv, fit_intercept=True, max_iter=10000)
    lasso.fit(Xtr_l, y_tr_adv)
    beta      = lasso.coef_ / scaler_l.scale_
    alpha_hat = lasso.intercept_ - (scaler_l.mean_ / scaler_l.scale_ * lasso.coef_).sum()
    y_pred_te = F_adv_te.values @ beta + alpha_hat
    y_te_adv  = P_test.loc[F_adv_te.index, port].values
    oos_results[port]["LASSO+"] = oos_r2(y_te_adv, y_pred_te)

    # ── Rolling OLS ───────────────────────────────────────────────────
    # Fit on all train, predict test with last 252-day betas
    y_all = P[port].values
    X_all = F.values
    n_tr  = len(train_idx)
    y_hat_test = []
    for t in range(n_tr, n_tr + len(test_idx)):
        X_win    = np.column_stack([np.ones(252), X_all[t-252:t]])
        y_win    = y_all[t-252:t]
        coef, *_ = np.linalg.lstsq(X_win, y_win, rcond=None)
        y_hat_test.append(np.dot(np.array([1] + list(X_all[t])), coef))
    oos_results[port]["Rolling"] = oos_r2(y_test, np.array(y_hat_test))

# ── Results table ─────────────────────────────────────────────────────────
print("=" * 70)
print("  OUT-OF-SAMPLE R² (test set: 2023-2025)")
print("=" * 70)
print(f"\n{'Portfolio':<22} {'OLS':>7} {'Ridge-WLS':>10} "
      f"{'LASSO+':>8} {'Rolling':>9} {'Best OOS':>10}")
print("-" * 68)

for port in P.columns:
    name = port.replace("portfolio_", "")
    r    = oos_results[port]
    best = max(r, key=r.get)
    print(f"  {name:<20} {r['OLS']:>7.3f} {r['Ridge-WLS']:>10.3f} "
          f"{r['LASSO+']:>8.3f} {r['Rolling']:>9.3f} {best:>10}")

print("\n── In-sample vs Out-of-sample gap ──")
print(f"\n{'Portfolio':<22} {'Best IS R²':>11} {'Best OOS R²':>12} {'Gap':>8}")
print("-" * 55)
for port in P.columns:
    name    = port.replace("portfolio_", "")
    r       = oos_results[port]
    best_oos = max(r.values())
    # Best in-sample from comparison table
    is_scores = {
        "OLS":       ols_results[port].rsquared,
        "Ridge-WLS": ridge_results[port]["r2"],
        "LASSO+":    lasso_advanced[port]["r2"],
        "Rolling":   rolling_ols[port]["r2"],  # defined in section 3.6
    }
    best_is = max(is_scores.values())
    gap     = best_oos - best_is
    print(f"  {name:<20} {best_is:>11.3f} {best_oos:>12.3f} {gap:>+8.3f}")

**Rolling OLS wins on every portfolio out-of-sample.** Time-varying betas adapt to the 2023–2025 post-hike regime; static models are anchored to a different environment.

`LASSO+` fails on `trend_neutral` (OOS R²≈0) — fitted on training data only, LASSO zeroes all features. The regularisation correctly identifies no generalisable signal. This was invisible in-sample; OOS surfaces it cleanly.

`inverse_vol` OOS exceeds its in-sample R² (+0.036) — the 2023–2025 rate dynamics happen to be well-captured by the factor structure.


### 3.9 — Model Comparison


In [ ]:
print(f"{'Portfolio':<22}  {'OLS':>6}  {'Ridge':>7}  {'LASSO+':>8}  {'Rolling':>9}  {'Huber':>7}  {'Best IS':>9}")
print("-" * 75)
for port in P.columns:
    name = port.replace("portfolio_", "")
    scores = {
        "OLS":     ols_results[port].rsquared,
        "Ridge":   ridge_results[port]["r2"],
        "LASSO+":  lasso_advanced[port]["r2"],
        "Rolling": rolling_ols[port]["r2"],
        "Huber":   huber_results[port]["r2"],
    }
    best = max(scores, key=scores.get)
    print(f"  {name:<20}  {scores['OLS']:>6.3f}  {scores['Ridge']:>7.3f}  "
          f"{scores['LASSO+']:>8.3f}  {scores['Rolling']:>9.3f}  "
          f"{scores['Huber']:>7.3f}  {best:>9}")

print("")
print("In-sample best:  equal_weight=Ridge | inverse_vol=Rolling | trend=LASSO+ | trend_neutral=Rolling")
print("Out-of-sample:   Rolling OLS wins on all four portfolios")
print("run_pipeline():  Rolling OLS as default (best OOS, no hyperparameters beyond window size)")

## 4. Performance Measurement

We evaluate the Ridge-WLS model using residual diagnostics and rolling stability analysis.


### 4.1 — Residual Diagnostics


In [ ]:
# ── Section 4 : Performance Measurement ──────────────────────────────────
# Statistical tests on Ridge-WLS residuals
from statsmodels.stats.stattools import jarque_bera
from statsmodels.stats.diagnostic import het_breuschpagan, acorr_ljungbox
import statsmodels.api as sm

print("=" * 75)
print("  RESIDUAL DIAGNOSTICS")
print("=" * 75)
print(f"\n{'Portfolio':<22} {'R²':>6} {'Adj R²':>8} {'JB p-val':>10} "
      f"{'LB(10) p':>10} {'BP p-val':>10}")
print("-" * 70)

diag_results = {}
n_factors = F.shape[1]

for port, res in ridge_results.items():
    resid  = res["residuals"]
    n      = len(resid)
    k      = n_factors + 1

    # Adjusted R-squared
    adj_r2 = 1 - (1 - res["r2"]) * (n - 1) / (n - k - 1)

    # Jarque-Bera: normality
    jb_stat, jb_p, _, _ = jarque_bera(resid)

    # Ljung-Box: autocorrelation
    lb_p = acorr_ljungbox(resid, lags=[10],
                          return_df=True)["lb_pvalue"].values[0]

    # Breusch-Pagan: heteroskedasticity
    X_const = sm.add_constant(F.values)
    bp_stat, bp_p, _, _ = het_breuschpagan(resid, X_const)

    name = port.replace("portfolio_", "")
    print(f"  {name:<20} {res['r2']:>6.3f} {adj_r2:>8.3f} "
          f"{jb_p:>10.4f} {lb_p:>10.4f} {bp_p:>10.4f}")

    diag_results[port] = {
        "adj_r2": adj_r2, "jb_p": jb_p,
        "lb_p": lb_p,     "bp_p": bp_p
    }

print("\nLegend:")
print("  JB  : Jarque-Bera  — H0: residuals are normal")
print("  LB  : Ljung-Box    — H0: no autocorrelation in residuals")
print("  BP  : Breusch-Pagan — H0: residuals are homoskedastic")
print("  p < 0.05 → reject H0")

In [ ]:
# ── Residual plots ────────────────────────────────────────────────────────
fig, axes = plt.subplots(len(ridge_results), 3,
                          figsize=(16, 4 * len(ridge_results)))

for i, (port, res) in enumerate(ridge_results.items()):
    resid  = res["residuals"]
    y_hat  = res["y_hat"]
    y_true = P[port].values
    name   = port.replace("portfolio_", "")
    dates  = F.index

    # ── Plot 1 : Residuals over time ──────────────────────────────────
    axes[i,0].plot(dates, resid, linewidth=0.8,
                   color="#3498db", alpha=0.8)
    axes[i,0].axhline(0, color="black", linewidth=0.8,
                      linestyle="--", alpha=0.5)
    axes[i,0].set_title(f"{name} — Residuals over time",
                        fontsize=10, fontweight="bold")
    axes[i,0].set_ylabel("Residual")
    axes[i,0].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    # ── Plot 2 : Fitted vs Actual ─────────────────────────────────────
    axes[i,1].scatter(y_hat, y_true, alpha=0.2,
                      s=5, color="#2ecc71")
    # 45-degree reference line
    mn = min(y_hat.min(), y_true.min())
    mx = max(y_hat.max(), y_true.max())
    axes[i,1].plot([mn, mx], [mn, mx], color="red",
                   linewidth=1.2, linestyle="--", label="perfect fit")
    axes[i,1].set_title(f"{name} — Fitted vs Actual",
                        fontsize=10, fontweight="bold")
    axes[i,1].set_xlabel("Fitted"); axes[i,1].set_ylabel("Actual")
    axes[i,1].legend(fontsize=8)

    # ── Plot 3 : Residual distribution vs Normal ──────────────────────
    axes[i,2].hist(resid, bins=80, density=True,
                   color="#9b59b6", alpha=0.7,
                   edgecolor="white", linewidth=0.3)
    # Normal reference
    x_range = np.linspace(resid.min(), resid.max(), 200)
    normal  = (1 / (resid.std() * np.sqrt(2 * np.pi)) *
               np.exp(-0.5 * ((x_range - resid.mean()) /
                               resid.std()) ** 2))
    axes[i,2].plot(x_range, normal, color="red",
                   linewidth=1.5, label="Normal distribution")
    axes[i,2].set_title(f"{name} — Residual distribution",
                        fontsize=10, fontweight="bold")
    axes[i,2].legend(fontsize=8)

plt.suptitle("Residual Diagnostics — Ridge-WLS",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

**JB p=0.000** everywhere — fat tails, expected. Standard errors not reliable for inference.

**Ljung-Box:** passive portfolios (p=0.33, 0.23) show no autocorrelation — model captures the linear structure. Trend portfolios (p=0.029, 0.0006) have significant autocorrelation — the static model misses time-dependent dynamics. Consistent with the rolling OLS outperformance.

**Breusch-Pagan p=0.000** everywhere — heteroskedasticity confirmed, validating WLS.


### 4.2 — Rolling R-squared Stability


In [ ]:
# ── Rolling R² (252-day window) ───────────────────────────────────────────
WINDOW = 252
fig, axes = plt.subplots(len(ridge_results), 1,
                          figsize=(13, 14), sharex=True)

for i, (port, res) in enumerate(ridge_results.items()):
    y        = P[port].values
    name     = port.replace("portfolio_", "")
    r2_roll  = []
    dates_r2 = []

    for start in range(0, len(F) - WINDOW):
        end  = start + WINDOW
        Xw   = np.column_stack([np.ones(WINDOW), F.values[start:end]])
        yw   = y[start:end]
        coef, *_ = np.linalg.lstsq(Xw, yw, rcond=None)
        yhat = Xw @ coef
        ss_r = np.sum((yw - yhat)**2)
        ss_t = np.sum((yw - yw.mean())**2)
        r2_roll.append(1 - ss_r/ss_t if ss_t > 0 else np.nan)
        dates_r2.append(F.index[end])

    r2_roll  = np.array(r2_roll)
    dates_r2 = np.array(dates_r2)

    # Line + shaded zones
    axes[i].plot(dates_r2, r2_roll, linewidth=1.5,
                 color="#2c3e50", zorder=3)
    axes[i].fill_between(dates_r2, r2_roll, 0.5,
                         where=(r2_roll >= 0.5),
                         color="#27ae60", alpha=0.4, label="R² > 0.5 (good)")
    axes[i].fill_between(dates_r2, r2_roll, 0,
                         where=((r2_roll >= 0) & (r2_roll < 0.5)),
                         color="#f39c12", alpha=0.4, label="0 < R² < 0.5 (moderate)")
    axes[i].fill_between(dates_r2, r2_roll, 0,
                         where=(r2_roll < 0),
                         color="#e74c3c", alpha=0.5, label="R² < 0 (poor)")

    axes[i].axhline(0.5, color="#27ae60", linewidth=0.8,
                    linestyle="--", alpha=0.7)
    axes[i].axhline(0,   color="black",   linewidth=0.8)

    for label, date in [("COVID",     "2020-03-20"),
                        ("Rate hike", "2022-01-01")]:
        axes[i].axvline(pd.Timestamp(date), color="#e74c3c",
                        linewidth=1.0, linestyle="--", alpha=0.6)
        axes[i].text(pd.Timestamp(date), 1.0, label,
                     fontsize=7.5, ha="center", color="#e74c3c")

    axes[i].set_title(f"{name} — Rolling 1-Year R²",
                      fontsize=11, fontweight="bold")
    axes[i].set_ylabel("R²", fontsize=9)
    axes[i].set_ylim(-0.15, 1.05)
    axes[i].legend(fontsize=8, loc="lower right", ncol=3)

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.suptitle("Section 4 — Rolling R² Stability (252-day window)",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

`equal_weight` stays above 0.7 throughout — reliable in every regime. The trend portfolios are volatile: R-squared spikes during crisis periods (macro factors briefly dominate trend returns) then falls back. This confirms Rolling OLS as the default model in `run_pipeline()`.


### 4.3 — Rolling Factor Exposures


In [ ]:
# ── Rolling Betas (252-day window) — equal_weight & inverse_vol ───────────
WINDOW   = 252
PORT_SHOW = ["portfolio_equal_weight", "portfolio_inverse_vol"]

fig, axes = plt.subplots(len(PORT_SHOW), 1,
                          figsize=(13, 5 * len(PORT_SHOW)),
                          sharex=True)

for i, port in enumerate(PORT_SHOW):
    y       = P[port].values
    name    = port.replace("portfolio_", "")
    betas_roll = {col: [] for col in F.columns}
    dates_roll = []

    for start in range(0, len(F) - WINDOW):
        end  = start + WINDOW
        Xw   = np.column_stack([np.ones(WINDOW), F.values[start:end]])
        yw   = y[start:end]
        coef, *_ = np.linalg.lstsq(Xw, yw, rcond=None)
        for j, col in enumerate(F.columns):
            betas_roll[col].append(coef[j + 1])
        dates_roll.append(F.index[end])

    for col in F.columns:
        axes[i].plot(dates_roll, betas_roll[col],
                     linewidth=1.2, label=col, alpha=0.85)

    axes[i].axhline(0, color="black", linewidth=0.8,
                    linestyle="--", alpha=0.4)
    for label, date in [("COVID",     "2020-03-20"),
                        ("Rate hike", "2022-01-01")]:
        axes[i].axvline(pd.Timestamp(date), color="#e74c3c",
                        linewidth=0.8, linestyle="--", alpha=0.5)
        axes[i].text(pd.Timestamp(date), axes[i].get_ylim()[1],
                     label, fontsize=7.5,
                     ha="center", color="#e74c3c")

    axes[i].set_title(f"{name} — Rolling Factor Betas (252d)",
                      fontsize=11, fontweight="bold")
    axes[i].set_ylabel("Beta")
    axes[i].legend(fontsize=8, ncol=4, loc="upper left")
    axes[i].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

plt.suptitle("Rolling Factor Exposures — Passive Portfolios",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

`equal_weight`: the `commodities` beta (~0.45) is structurally stable over 10 years — static model appropriate. `inverse_vol`: the `interest_rate` beta evolves substantially, especially around 2022 — rolling estimation is the better choice.


## 5. PCA Factor Construction

*To be completed.*


## 6. Trend Portfolio

*To be completed.*


## 7. Reusable Pipeline

*To be completed.*


In [ ]:
def run_pipeline(data_path):
    """
    Full factor analysis pipeline.
    Accepts any folder with factors.csv, assets.csv, portfolios.csv.
    Returns dict with keys: exposures, performance, pca, trend.
    """
    dataset    = load_dataset(data_path)
    factors_   = dataset["factors"]
    assets_    = dataset["assets"]
    portfolios_= dataset["portfolios"]
    results    = {
        "exposures":   None,
        "performance": None,
        "pca":         None,
        "trend":       None,
    }
    return results